In [1]:
import sys
print(sys.executable)

c:\Users\NIHAL\Desktop\genai_assignment\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

dataset_path = project_root / "data" / "raw" / "LF Jobs.xlsx"

df = pd.read_excel(dataset_path)

print("Dataset shape:", df.shape)
display(df.head())
df.info()

Dataset shape: (1000, 9)


,ID,Job Category,Job Title,Company Name,Publication Date,Job Location,Job Level,Tags,Job Description
0,LF0001,Data and Analytics,"DIR, Equities Quant",Merrill,2025-07-28T23:00:54Z,"New York, NY",Mid Level,NaN,<p><b>Job Description:</b><br><br>At Bank of A...
1,LF0002,Data and Analytics,Lead Administrator - L1,Wipro,2025-06-15T23:43:28Z,"Hyderabad, India",Senior Level,NaN,"<p>Wipro Limited (NYSE: WIT, BSE: 507685, NSE:..."
2,LF0003,Data and Analytics,Principal DEX Solutions Architect,Eaton,2025-07-30T11:38:27Z,"Chicago, IL",Senior Level,NaN,<p>Eaton's Digital Employee Experience (DEX) t...
3,LF0004,Data and Analytics,Retail Sales Supervisor- Melrose Studio,Soho House,2025-07-29T23:31:48Z,"West Hollywood, CA",Senior Level,NaN,<p><strong>The Role…</strong></p>\n\n<p>We are...
4,LF0005,Data and Analytics,CAT Program Specialist,Royal Bank of Canada,2024-05-03T11:42:19Z,"Minneapolis, MN",Mid Level,NaN,<p><b>Job Summary</b><br><br><b> Job Descripti...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   ID                1000 non-null   object
 1   Job Category      1000 non-null   object
 2   Job Title         1000 non-null   object
 3   Company Name      1000 non-null   object
 4   Publication Date  1000 non-null   object
 5   Job Location      991 non-null    object
 6   Job Level         1000 non-null   object
 7   Tags              120 non-null    object
 8   Job Description   1000 non-null   object
dtypes: object(9)
memory usage: 70.4+ KB


## Initial dataset inspection

The dataset contains 1,000 job listings and nine columns.
Job Location has nine missing values, while Tags has 880 missing
values. All job descriptions are populated and contain HTML.
Publication Date requires datetime parsing.

Some category–title combinations warrant inspection. Original
values will be preserved unless there is evidence supporting a correction.

In [3]:
missing_counts = df.isna().sum()
display(missing_counts)

duplicate_rows = df.duplicated().sum()
print("Fully duplicated rows:", duplicate_rows)

duplicate_ids = df["ID"].duplicated().sum()
print("Repeated job IDs:", duplicate_ids)

ID                    0
Job Category          0
Job Title             0
Company Name          0
Publication Date      0
Job Location          9
Job Level             0
Tags                880
Job Description       0
dtype: int64

Fully duplicated rows: 0
Repeated job IDs: 0


### Missing values and duplicate checks

No fully duplicated rows or repeated job IDs were found.
Nine listings have missing locations, and 880 have missing tags.
All records are retained at this stage because job descriptions
are populated. Missing metadata will not be treated as known information.

In [4]:
category_counts = df["Job Category"].value_counts(dropna=False)
display(category_counts)

level_counts = df["Job Level"].value_counts(dropna=False)
display(level_counts)

company_count = df["Company Name"].nunique()
print("Unique companies:", company_count)

Job Category
Data and Analytics           166
Software Engineering         166
Design and UX                166
Sales                        166
Project Management           166
Advertising and Marketing    166
General                        4
Name: count, dtype: int64

Job Level
Senior Level    610
Mid Level       349
Internship       24
Entry Level      17
Name: count, dtype: int64

Unique companies: 145


### Dataset coverage

The dataset contains seven categories and 145 unique company names.
Six categories have 166 listings each, while General has four.

Senior Level accounts for 610 listings, followed by Mid Level (349),
Internship (24), and Entry Level (17). This uneven coverage should
be considered when evaluating queries for different experience levels.

In [5]:
parsed_dates = pd.to_datetime(
    df["Publication Date"],
    errors="coerce",
    utc=True
)

invalid_dates = parsed_dates.isna().sum()
print("Missing or unparseable dates:", invalid_dates)

print("Earliest publication date:", parsed_dates.min())
print("Latest publication date:", parsed_dates.max())

Missing or unparseable dates: 0
Earliest publication date: 2023-02-01 10:05:24+00:00
Latest publication date: 2025-08-01 00:19:55+00:00


### Publication dates

All 1,000 publication dates parsed successfully as UTC timestamps.
Dates range from February 1, 2023, to August 1, 2025.
The dataset is historical; publication dates do not confirm
whether listings remain open.

In [6]:
sample_job = df.iloc[0]

print("Job ID:", sample_job["ID"])
print("Job title:", sample_job["Job Title"])
print("\nOriginal description:\n")
print(sample_job["Job Description"])

Job ID: LF0001
Job title: DIR, Equities Quant

Original description:

<p><b>Job Description:</b><br><br>At Bank of America, we are guided by a common purpose to help make financial lives better through the power of every connection. We do this by driving Responsible Growth and delivering for our clients, teammates, communities and shareholders every day.<br><br>Being a Great Place to Work is core to how we drive Responsible Growth. This includes our commitment to being an inclusive workplace, attracting and developing exceptional talent, supporting our teammates' physical, emotional, and financial wellness, recognizing and rewarding performance, and how we make an impact in the communities we serve.<br><br>Bank of America is committed to an in-office culture with specific requirements for office-based attendance and which allows for an appropriate level of flexibility for our teammates and businesses based on role-specific considerations.<br><br>At Bank of America, you can build a succ

In [7]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(sample_job["Job Description"], "html.parser")

cleaned_description = soup.get_text(separator="\n", strip=True)

print(cleaned_description)

Job Description:
At Bank of America, we are guided by a common purpose to help make financial lives better through the power of every connection. We do this by driving Responsible Growth and delivering for our clients, teammates, communities and shareholders every day.
Being a Great Place to Work is core to how we drive Responsible Growth. This includes our commitment to being an inclusive workplace, attracting and developing exceptional talent, supporting our teammates' physical, emotional, and financial wellness, recognizing and rewarding performance, and how we make an impact in the communities we serve.
Bank of America is committed to an in-office culture with specific requirements for office-based attendance and which allows for an appropriate level of flexibility for our teammates and businesses based on role-specific considerations.
At Bank of America, you can build a successful career with opportunities to learn, grow, and make an impact. Join us!
Job Description:
This job is r

In [8]:
def clean_description(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator="\n", strip=True)

df["Clean Description"] = df["Job Description"].apply(clean_description)

empty_descriptions = df["Clean Description"].str.strip().eq("").sum()
print("Empty descriptions after cleaning:", empty_descriptions)

display(df[["ID", "Job Title", "Clean Description"]].sample(3, random_state=42))

Empty descriptions after cleaning: 0


,ID,Job Title,Clean Description
521,LF0522,Part-Time Beauty Advisor - Sephora,"About the Role\nAs Beauty Advisor, you will dr..."
737,LF0738,KYC Specialist,This is Adyen\nWe took an unobvious approach t...
740,LF0741,Senior Project Manager,"Who We Are\nAt Kyndryl, we design, build, mana..."


### HTML cleaning

Descriptions were parsed with BeautifulSoup and extracted as plain
text, using newlines to separate text elements. Original HTML was
preserved in a separate column. No descriptions became empty.
A full example was inspected to check that key job details remained.

In [9]:
description_word_counts = df["Clean Description"].str.split().str.len()

display(description_word_counts.describe())

longest_index = description_word_counts.idxmax()
print("Longest description job ID:", df.loc[longest_index, "ID"])
print("Longest description word count:", description_word_counts.loc[longest_index])

count    1000.000000
mean      752.366000
std       269.820832
min        21.000000
25%       575.000000
50%       736.500000
75%       933.750000
max      2240.000000
Name: Clean Description, dtype: float64

Longest description job ID: LF0859
Longest description word count: 2240


In [10]:
shortest_index = description_word_counts.idxmin()

print("Job ID:", df.loc[shortest_index, "ID"])
print("Job title:", df.loc[shortest_index, "Job Title"])
print("Word count:", description_word_counts.loc[shortest_index])
print("\nClean description:\n")
print(df.loc[shortest_index, "Clean Description"])

Job ID: LF0204
Job title: Software Design Engineer
Word count: 21

Clean description:

岗位职责
负责开发基于C++/C# 变频器伺服PLC上位机软件的开发及调试.根据项目需求完成相应模块的设计和开发.根据任务分配提交高质量代码并按时完成目标.具有良好的沟通能力协作能力及分析问题的能力.软件开发工程中完成相应文档的编写与修改.熟悉git.
任职要求
本科及以上学历具有5年以上独立开发上位机软件的经验;熟悉常用的通讯协议如ModbusEtherCAT等;有工业自动化类产品(PLC, HMI, 变频器)开发经验者优先考虑;熟悉CODESYS者优先具有良好的英文读写能力, 能看懂英文技术文档;较好逻辑思维善于沟通能够解决问题具有团队合作精神,积极主动
资格
岗位职责
负责开发基于C++/C# 变频器伺服PLC上位机软件的开发及调试.根据项目需求完成相应模块的设计和开发.根据任务分配提交高质量代码并按时完成目标.具有良好的沟通能力协作能力及分析问题的能力.软件开发工程中完成相应文档的编写与修改.熟悉git.
任职要求
本科及以上学历具有5年以上独立开发上位机软件的经验;熟悉常用的通讯协议如ModbusEtherCAT等;有工业自动化类产品(PLC, HMI, 变频器)开发经验者优先考虑;熟悉CODESYS者优先具有良好的英文读写能力, 能看懂英文技术文档;较好逻辑思维善于沟通能够解决问题具有团队合作精神,积极主动
时间表:
全职
请求编号:
009G65


### Multilingual content

Inspection of LF0204 revealed a Chinese-language description.
Its low whitespace-separated word count does not indicate little
content, because Chinese generally does not use spaces between words.
Retain the listing and consider multilingual support when selecting
embedding and reranking models. Use model token counts when checking
input limits.

In [11]:
contains_cjk = df["Clean Description"].str.contains(
    r"[\u4e00-\u9fff]",
    regex=True,
    na=False
)

print("Descriptions containing CJK characters:", contains_cjk.sum())

display(
    df.loc[contains_cjk, ["ID", "Job Title", "Job Location"]]
)

Descriptions containing CJK characters: 11


,ID,Job Title,Job Location
203,LF0204,Software Design Engineer,"Shanghai, China, Wuxi, China"
276,LF0277,SAP Consultant-Energy Environment and Utilities,"Sapporo, Japan"
303,LF0304,SAP領域のAMSプロジェクトマネジャー,"Tokyo, Japan"
478,LF0479,"Electronics Hardware, Senior Principal Architect","Shanghai, China"
584,LF0585,Sales Representative APM Tokyo（担当：東京、埼玉、群馬、栃木、山梨）,"Tokyo, Japan"
684,LF0685,Project Manager-Energy Environment and Utiities,"Sendai, Japan"
687,LF0688,Quality Leader,"Shanghai, China"
700,LF0701,【KJTS】プロジェクトマネージャー（サービスマネジメント）,"Flexible / Remote, Tokyo, Japan"
736,LF0737,Project Manager for Financial Services,"Tokyo, Japan"
740,LF0741,Senior Project Manager,"Tokyo, Japan"


### Multilingual character check

Eleven descriptions contain characters in the checked CJK
ideograph range. This is a character-based check, not complete
language detection. All listings are retained. Model selection
and chunking must account for multilingual content.

In [12]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "jobs_cleaned.jsonl"

df.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved to:", output_path)

Saved to: c:\Users\NIHAL\Desktop\genai_assignment\data\processed\jobs_cleaned.jsonl


In [13]:
reloaded_df = pd.read_json(
    output_path,
    orient="records",
    lines=True,
    convert_dates=False
)

pd.testing.assert_frame_equal(df, reloaded_df)

print("Verification passed: saved data matches the DataFrame.")

Verification passed: saved data matches the DataFrame.


C:\Users\NIHAL\AppData\Local\Temp\ipykernel_556\4070424186.py:8: FutureWarning: Mismatched null-like values nan and None found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(df, reloaded_df)


In [14]:
original_check = df.fillna("")
reloaded_check = reloaded_df.fillna("")

pd.testing.assert_frame_equal(original_check, reloaded_check)

print("Verification passed: values match after normalizing missing values.")

Verification passed: values match after normalizing missing values.


In [15]:
from transformers import AutoTokenizer

embedding_model_name = "intfloat/multilingual-e5-small"

tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

print("Tokenizer loaded:", embedding_model_name)
print("Tokenizer input limit:", tokenizer.model_max_length)

Tokenizer loaded: intfloat/multilingual-e5-small
Tokenizer input limit: 512


In [16]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

cleaned_path = project_root / "data" / "processed" / "jobs_cleaned.jsonl"

df = pd.read_json(cleaned_path, lines=True, convert_dates=False)

print("Loaded shape:", df.shape)

Loaded shape: (1000, 10)


In [17]:
def count_tokens(text):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )
    return len(token_ids)

description_token_counts = df["Clean Description"].apply(count_tokens)

display(description_token_counts.describe())

print(
    "Descriptions exceeding 512 tokens:",
    (description_token_counts > 512).sum()
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (786 > 512). Running this sequence through the model will result in indexing errors


count    1000.000000
mean     1098.341000
std       387.406883
min       174.000000
25%       851.000000
50%      1063.000000
75%      1361.250000
max      3266.000000
Name: Clean Description, dtype: float64

Descriptions exceeding 512 tokens: 947


## Token lengths and chunking plan

Using the multilingual-e5-small tokenizer, cleaned descriptions
average 1,098.3 content tokens and range from 174 to 3,266 tokens.
947 descriptions exceed the model's 512-token input limit,
before adding prefixes, metadata, or special tokens.

We will initially target 384 description tokens per chunk with
48 tokens of overlap. Final embedding inputs will be checked
against the 512-token limit after adding metadata and prefixes.
These settings are starting points to evaluate, not optimized values.

In [18]:
MAX_INPUT_TOKENS = 512
CHUNK_SIZE = 384
CHUNK_OVERLAP = 48

assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE

print("Chunk step:", CHUNK_SIZE - CHUNK_OVERLAP)

Chunk step: 336
